# Mitr Dataset #
## Finding the number of comments wrongly detected as pure-English ##

This section finds the number of comments wrongly detected as pure-English for methods: Current Method, Alternative 1 (sliding window with fixed list), Alternative 2 (sliding window with geo priors), and Alternative 3 (no sliding window with geo priors. )

In [2]:
import polars as pl
import os
 
from lingua import Language, LanguageDetectorBuilder
 
languages = [Language.ENGLISH, Language.FRENCH, Language.GERMAN, Language.SPANISH, \
             Language.ITALIAN, Language.CHINESE, Language.JAPANESE, Language.PORTUGUESE, \
             Language.INDONESIAN, Language.THAI, Language.MALAY, Language.ARABIC, \
             Language.POLISH, Language.CZECH]
# detector = LanguageDetectorBuilder.from_languages(*Language.all()).build()
detector = LanguageDetectorBuilder.from_languages(*languages).with_minimum_relative_distance(0.05).build()
 
# Define a function to detect language and return the ISO code
def _detect_language_iso(text: str) -> str:
    if text is None or text.strip() == "":
        return "unknown"
    lang = detector.detect_language_of(text)
    return lang.iso_code_639_1.name.lower() if lang else "unknown"
 
def _detect_multiple_language_iso(text: str) -> list[str]:
    if text is None or text.strip() == "":
        return ["unknown"]
    langs = detector.detect_multiple_languages_of(text)
    results = list(set([lang.language.iso_code_639_1.name.lower() for lang in langs] or ["unknown"]))
    return results
 
def df_language_verified(df: pl.LazyFrame) -> pl.LazyFrame:
    # Load the LazyFrame and add the language detection column
    # lf = df.with_columns([
    #         pl.col("comments").map_elements(_detect_language_iso, return_dtype=pl.String).alias("comments_language_id")
    #     ])
   
    lf = df.with_columns([
            pl.col("comments").map_elements(_detect_multiple_language_iso, return_dtype=pl.List(pl.Utf8)).alias("comments_language_id")
        ])
 
    return lf
 
file = "Mitr Phol JUN 2025- text comments"
 
# df = pl.scan_csv(f'./data/src/{file}.csv')
 
df = pl.read_excel(f'../data/{file}.xlsx.xlsx').lazy()
df = df.unique(subset=["comments"], maintain_order=True)
 
# df_language_verified(df).sink_csv(f'./data/src/{file}_detected.csv')
 
# df_language_verified(df).collect().to_pandas().to_excel(f'./data/src/{file}_detected.xlsx', index=False)
 
df_language_verified(df).sink_parquet(f'../data/{file}_multiple_detected.parquet')
 
# print(_detect_multiple_language_iso("Sudah baik tetapi harus terus ditingkatkan"))

Could not determine dtype for column 5, falling back to string
Could not determine dtype for column 6, falling back to string
Could not determine dtype for column 7, falling back to string
Could not determine dtype for column 8, falling back to string
Could not determine dtype for column 9, falling back to string
Could not determine dtype for column 10, falling back to string


In [3]:
import pandas as pd
df = pd.read_parquet(f'../data/{file}_multiple_detected.parquet').rename(columns={"comments_language_id": "curr"})
df = df.loc[:, ["comments", "curr"]]
df["curr"] = df["curr"].apply(lambda x: list(x))
display(df)

,comments,curr
0,-,[unknown]
1,อยากให้มีการนำ AI เข้ามาใช้งาน ทั้งที่เป็น AI ...,[th]
2,มีทางเลือกเพื่อใช้ในการตัดสินใจมากขึ้น ลดเวลาก...,[th]
3,เพิ่มประสิทธิภาพการทำงานให้กับพนักงานให้ทำงานไ...,[th]
4,มีความทันสมัยมากขึ้น,[th]
...,...,...
432,เพิ่มช่องทางการศึกษาเกี่ยวกับ AIให้มากขึ้น,[th]
433,-เร่งนำ AI มาช่วยในการลดงานที่ซ้ำซ้อน เช่นการว...,"[th, pl]"
434,อยากให้เทคโนโลยีเข้ามาช่วยในการทำงานให้ง่าย สะ...,[th]
435,การนำ AI มาช่วยพัฒนาระบบทำใสของน้ำอ้อย,[th]


In [8]:
display(df.loc[df["curr"].apply(lambda x: list(x) == ["en"]), ["comments", "curr"]])

,comments,curr
239,Should have AI course started from basic to ad...,[en]
353,นอกจากการใช้AI การจัดเตรียมข้อมูลสำหรับการGene...,[en]


## Current Method Manual Review Report ##
Results above are filtered to only have comments detected as only English. Then, each comment is manually reviewed whether it actually is written in only English. Among 2 comments detected as only English, 1 comment is wrong.

## Alternative Method 1: Sliding Window, Fixed List ##

In [10]:
from lingua import Language
from lingua import LanguageDetectorBuilder

def window_sliders(text:str, size:int) -> list[str]:
    """
    Generates a list of sliding windows (substrings) of a given word length from the input text.

    Parameters:
        text (str): The input string to slide over.
        size (int): The number of words in each sliding window.

    Returns:
        list[str]: A list of strings, each containing `size` consecutive words from the input text.

    Raises:
        ValueError: If `text` is not a string.
        ValueError: If `size` is not an integer.
        ValueError: If `size` is less than the number of words in the text.

    Example:
        >>> window_sliders("the quick brown fox jumps", 3)
        ['the quick brown', 'quick brown fox', 'brown fox jumps']
    """
    
    if not isinstance(text,str): raise ValueError("Text must be a string")
    if not isinstance(size,int): raise ValueError("Size must be an int")
    if size > len(text.split()): 
        return [text]
        # raise ValueError("Size must be greater than number of words in the text.")

    result = []
    text = text.split()
    last_idx = len(text) - size
    for i in range(0,last_idx+1):
        result.append((" ").join(text[i:i+size]))
    
    return result

COUNTRY_TO_LANGS = {
    "THAILAND": ["THAI", "CHINESE", "ENGLISH"],
    "SPAIN": ["SPANISH", "CATALAN", "BASQUE", "ENGLISH"],
    "INDIA": ["HINDI", "BENGALI", "MARATHI", "TAMIL", "TELUGU", "GUJARATI", "URDU", "ENGLISH"],
    "INDONESIA": ["INDONESIAN", "ENGLISH"],
    "JAPAN": ["JAPANESE", "ENGLISH", "CHINESE", "KOREAN"],
    "MALAYSIA": ["MALAY", "ENGLISH", "CHINESE", "TAMIL"],
}

def get_langs(country_to_langs:dict, country):
    """
    Retrieve a list of Language objects corresponding to the languages spoken in a given country.

    Args:
        country_to_langs (dict): A dictionary mapping country names (str) to lists of language codes or names (str).
        country (str): The name of the country for which to retrieve languages.

    Returns:
        list: A list of Language objects corresponding to the languages spoken in the specified country.

    Raises:
        ValueError: If the specified country does not exist in the country_to_langs dictionary.

    Example:
        >>> get_langs(COUNTRY_TO_LANGS, "INDIA")
        [Language.HINDI, Language.BENGALI, Language.MARATHI, Language.TAMIL, Language.TELUGU, Language.GUJARATI, Language.URDU, Language.ENGLISH]
    """

    if country not in country_to_langs.keys():
        raise ValueError("Country doesnt exist in the country_to_langs")
    return [Language.from_str(l) for l in country_to_langs[country]]

def detect_multi_lang(texts:list[str]|str, languages, t=.4, window=False, n=1) -> list:
    """
    Detects multiple languages present in a given text or a list of texts using the Lingua language detector.
    IMPORTANT: This function DEPENDS on the the `window_sliders` function to generate sliding windows for language detection.

    Parameters:
        texts (list[str] | str): A single string or a list of strings to detect languages from.
        languages (list): List of Language objects to consider for detection.
        t (float, optional): Minimum relative distance threshold for language detection. Default is 0.4.
        window (bool, optional): If True, uses sliding window approach for detection. Default is False.
        n (int, optional): Window size (number of words) for sliding window detection. Default is 1.

    Returns:
        list[list[IsoCode639_1]]: For a list of texts, returns a list of detected language ISO codes for each text.
        list[IsoCode639_1]: For a single string, returns a list of detected language ISO codes.

    Raises:
        ValueError: If input is not a string or list of strings.
        ValueError: If any element in the input list is not a string.

    Example:
        >>> detect_multi_lang("Parlez-vous français? Ich spreche Deutsch.", [Language.ENGLISH, Language.FRENCH, Language.GERMAN], t=0.5, window=True, n=3)
        [IsoCode639_1.FR, IsoCode639_1.DE]
    """

    detector = LanguageDetectorBuilder.from_languages(*languages).with_minimum_relative_distance(t).build()

    if isinstance(texts,list):
        if any(not isinstance(x,str) for x in texts):
            raise ValueError("The input text contains non-string item, it must only contains string.")
        
        # Use window mode
        if window:
            result = []
            for s in texts:
                one_result = detector.detect_multiple_languages_in_parallel_of(window_sliders(s,n))
                r = set()
                for l in one_result:
                    for i in l:
                        r.add(i.language.iso_code_639_1)
                result.append(list(r))
                
            return result

        result_list = detector.detect_multiple_languages_in_parallel_of(texts)
        result = []
        for l in result_list:
            temp = [r.language.iso_code_639_1 for r in l]
            result.append(list(set(temp)))
        
        return result
        
    elif isinstance(texts, str):
        result_set = detector.detect_multiple_languages_of(texts)

        result = [r.language.iso_code_639_1 for r in result_set]
        
        return list(set(result))
    else:
        raise ValueError("The input text must be either a list of strings or a string.")

In [12]:
# Import necessary libraries
import pandas as pd
from lingua import Language

# Define the languages to consider for detection
languages = [Language.ENGLISH, Language.FRENCH, Language.GERMAN, Language.SPANISH, \
             Language.ITALIAN, Language.CHINESE, Language.JAPANESE, Language.PORTUGUESE, \
             Language.INDONESIAN, Language.THAI, Language.MALAY, Language.ARABIC, \
             Language.POLISH, Language.CZECH]

# Detect languages in the comments using the defined function
detected_langs = detect_multi_lang(df["comments"].tolist(), languages, t=0.2, window=True, n=1)
detected_langs = [[l.name.lower() for l in r] for r in detected_langs]
df["sw_fl"] = detected_langs

df.loc[df["sw_fl"].apply(lambda x: x == ["en"]), ["comments", "sw_fl"]]

,comments,sw_fl
239,Should have AI course started from basic to ad...,[en]


## Alternative 1 Manual Review Report ##
Results above are filtered to only have comments detected as only English. Then, each comment is manually reviewed whether it actually is written in only English. Among 1 comment detected as only English, 0 comment is wrong.

## Alternative Method 2: Sliding Window, Geo Priors ##

In [14]:
# Import necessary libraries
import pandas as pd
from lingua import Language

# Detect languages in the comments using the defined function
detected_langs = detect_multi_lang(df["comments"].fillna("").tolist(), get_langs(COUNTRY_TO_LANGS, "THAILAND"), t=0.3, window=True, n=1)
detected_langs = [[l.name.lower() for l in r] for r in detected_langs]
df["sw_geo"] = detected_langs

# Filter the DataFrame to only include comments detected as English
df.loc[df["sw_geo"].apply(lambda x: x == ["en"]), ["comments", "sw_geo"]]

,comments,sw_geo
9,N/A,[en]
239,Should have AI course started from basic to ad...,[en]
359,I firmly believe that driving the organization...,[en]
409,OK,[en]


## Alternative 2 Manual Review Report ##
Results above are filtered to only have comments detected as only English. Then, each comment is manually reviewed whether it actually is written in only English. Among 4 comments detected as only English, 0 comment is wrong.

## Alternative Method 3: No Sliding Window, Geo Priors ##

In [16]:
# Import necessary libraries
import pandas as pd
from lingua import Language

# Detect languages in the comments using the defined function
detected_langs = detect_multi_lang(df["comments"].fillna("").tolist(), get_langs(COUNTRY_TO_LANGS, "THAILAND"), t=0.3, window=False)
detected_langs = [[l.name.lower() for l in r] for r in detected_langs]
df["no_sw_geo"] = detected_langs

# Filter the DataFrame to only include comments detected as English
df.loc[df["no_sw_geo"].apply(lambda x: x == ["en"]), ["comments", "no_sw_geo"]]

,comments,no_sw_geo
9,N/A,[en]
239,Should have AI course started from basic to ad...,[en]
333,1 แผนก : 1 AI Model,[en]
343,อยากให้มีการจัดอบรมเกี่ยวกับ AI&Digital ให้กับ...,[en]
353,นอกจากการใช้AI การจัดเตรียมข้อมูลสำหรับการGene...,[en]
359,I firmly believe that driving the organization...,[en]
409,OK,[en]
419,ใช้ chatGPT,[en]


## Alternative 3 Manual Review Report ##
Results above are filtered to only have comments detected as only English. Then, each comment is manually reviewed whether it actually is written in only English. Among 8 comment detected as pure English, 4 comments are wrong.

## Prepare detection results for side-by-side review ##

This section performs side-by-side review between detection results from the current method with the new method (sliding window with geo priors).

In [26]:
df["curr_lang_cat"] = df["curr"].apply(lambda x: "unknown" if x == ["unknown"] else "pure_english" if x == ["en"] else "mixed")
df["sw_fl_lang_cat"] = df["sw_fl"].apply(lambda x: "unknown" if x == [] else "pure_english" if x == ["en"] else "mixed")
df["sw_geo_lang_cat"] = df["sw_geo"].apply(lambda x: "unknown" if x == [] else "pure_english" if x == ["en"] else "mixed")
df["no_sw_geo_lang_cat"] = df["no_sw_geo"].apply(lambda x: "unknown" if x == [] else "pure_english" if x == ["en"] else "mixed")

summary = pd.DataFrame({
    "current": (df.groupby("curr_lang_cat").count())["comments"],
    "sw_geo": (df.groupby("sw_geo_lang_cat").count())["comments"],
    "no_sw_geo": (df.groupby("no_sw_geo_lang_cat").count())["comments"],
    "sw_fl": (df.groupby("sw_fl_lang_cat").count())["comments"],
})

display(summary)
display(df.loc[:,["comments","curr","sw_geo"]])

# Save the result for manual review
df.loc[:,["comments","curr","sw_geo"]].to_excel("../data/Mitr_detections_analysis.xlsx", index=False)

,current,sw_geo,no_sw_geo,sw_fl
mixed,424,426,422,427
pure_english,2,4,8,1
unknown,11,7,7,9


,comments,curr,sw_geo
0,-,[unknown],[]
1,อยากให้มีการนำ AI เข้ามาใช้งาน ทั้งที่เป็น AI ...,[th],"[en, th]"
2,มีทางเลือกเพื่อใช้ในการตัดสินใจมากขึ้น ลดเวลาก...,[th],[th]
3,เพิ่มประสิทธิภาพการทำงานให้กับพนักงานให้ทำงานไ...,[th],[th]
4,มีความทันสมัยมากขึ้น,[th],[th]
...,...,...,...
432,เพิ่มช่องทางการศึกษาเกี่ยวกับ AIให้มากขึ้น,[th],[th]
433,-เร่งนำ AI มาช่วยในการลดงานที่ซ้ำซ้อน เช่นการว...,"[th, pl]","[en, th]"
434,อยากให้เทคโนโลยีเข้ามาช่วยในการทำงานให้ง่าย สะ...,[th],"[en, th]"
435,การนำ AI มาช่วยพัฒนาระบบทำใสของน้ำอ้อย,[th],"[en, th]"


In [28]:

mitr_detections_analysis_result_df = pd.read_excel("../data/Mitr_detections_analysis_result.xlsx")
display(mitr_detections_analysis_result_df.groupby("acceptability").count().drop(columns=["comments", "curr"]).rename(columns={"sw_geo": "count"}))


,count
acceptability,
2.0,5


## Side-by-side Comparison Analysis Report ##

For each comment, I detect the languages using both methods and compare the results. If both methods cause the comment to be translated and it actually needs translation, I consider the results as acceptable. Also, if both methods cause the comment to skip translation and it actually does not need translation, I consider the results as acceptable.

But, if the results cause different actions (one causes translation while the other skip translation), I consider the results as unacceptable and document which method gives the correct answer (translate or not) in the table above. '0' means both results give the wrong action, '1' means the current method gives correct action, and '2' means the new method gives the correct action.

There are 5 comments flagged as unacceptable. Method 1 gives 0 right answers and method 2 gives 5 right answers.

Method 1's behaviour is quite unpredictable, sometimes it struggles to detect Thai segment although that segment is long.

In [6]:
t = pd.read_excel("../data/Mitr_detections_analysis_result.xlsx")
t[t["acceptability"] == 2].to_excel("../data/Mitr_detections_analysis_result_filtered.xlsx", index=False)